# 23_E15 - Traducción a backend/MVP

Este notebook transforma la cadena funcional E13/E14 en una estructura inicial de servicio Python para MVP.

**E15 no entrena modelos nuevos.** Su objetivo es pasar de notebooks experimentales a módulos reutilizables:

- configuración centralizada,
- contratos de entrada/salida,
- reglas de quality flags,
- reglas del agente/orquestador,
- API FastAPI inicial,
- documentación para backend/Codex,
- smoke test con outputs reales de E14.

La idea es dejar una base que luego pueda integrarse con el backend Spring Boot o ejecutarse como microservicio Python.


In [ ]:
from pathlib import Path
import json
import sys
import shutil
import textwrap
import importlib
import pandas as pd
import numpy as np

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

PFI_ROOT = Path("/content/drive/MyDrive/PFI_MVP")
REPO_ROOT = PFI_ROOT / "repo"

E14_ROOT = PFI_ROOT / "results" / "E14_ai_agent_orchestrator"
E15_ROOT = PFI_ROOT / "results" / "E15_backend_mvp_translation"
DOCS_ROOT = PFI_ROOT / "docs"

SERVICE_ROOT = REPO_ROOT / "ai_service"
PKG_ROOT = SERVICE_ROOT / "pfi_ai_service"

for p in [E15_ROOT, DOCS_ROOT, SERVICE_ROOT, PKG_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("PFI_ROOT:", PFI_ROOT)
print("REPO_ROOT:", REPO_ROOT, "| exists:", REPO_ROOT.exists())
print("SERVICE_ROOT:", SERVICE_ROOT)
print("E14_ROOT:", E14_ROOT, "| exists:", E14_ROOT.exists())
print("E15_ROOT:", E15_ROOT)


## 1. Generar archivos del servicio Python

Esta celda escribe una carpeta `ai_service/` dentro del repo. El objetivo no es reemplazar el backend Java, sino crear el servicio IA que el backend puede invocar.


In [ ]:
SERVICE_FILES = {'__init__.py': '"""PFI AI Service.\n\nMódulos de soporte para inferencia, quality flags y agente/orquestador IA.\n"""\n\n__version__ = "0.1.0"\n', 'settings.py': 'from __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom pathlib import Path\nimport os\n\n\n@dataclass(frozen=True)\nclass ServiceSettings:\n    """Rutas centralizadas del servicio IA.\n\n    En Colab se asume /content/drive/MyDrive/PFI_MVP.\n    En backend/MVP se puede configurar con la variable PFI_ROOT.\n    """\n\n    pfi_root: Path\n    models_root: Path\n    results_root: Path\n    figures_root: Path\n    docs_root: Path\n\n    sagittal_model_path: Path\n    axial_model_path: Path\n\n    e13_results_root: Path\n    e14_results_root: Path\n\n\ndef get_settings() -> ServiceSettings:\n    pfi_root = Path(os.getenv("PFI_ROOT", "/content/drive/MyDrive/PFI_MVP"))\n\n    models_root = pfi_root / "models"\n    results_root = pfi_root / "results"\n    figures_root = pfi_root / "figures"\n    docs_root = pfi_root / "docs"\n\n    return ServiceSettings(\n        pfi_root=pfi_root,\n        models_root=models_root,\n        results_root=results_root,\n        figures_root=figures_root,\n        docs_root=docs_root,\n        sagittal_model_path=models_root / "E12_sagittal_multiclass_final_best.pt",\n        axial_model_path=models_root / "E10_axial_t2_final_training_clean_best.pt",\n        e13_results_root=results_root / "E13_multiplanar_inference_pipeline",\n        e14_results_root=results_root / "E14_ai_agent_orchestrator",\n    )\n\n\nMODEL_REGISTRY = {\n    "sagittal_spider": {\n        "plane": "sagittal",\n        "num_classes": 4,\n        "class_names": {\n            0: "background",\n            1: "vertebra_group",\n            2: "canal",\n            3: "disc_group",\n        },\n        "human_review_required": True,\n    },\n    "axial_t2_alkafri": {\n        "plane": "axial",\n        "num_classes": 6,\n        "class_names": {\n            0: "background_250",\n            1: "raw_0",\n            2: "raw_50",\n            3: "raw_100",\n            4: "raw_150",\n            5: "raw_200",\n        },\n        "human_review_required": True,\n    },\n}\n', 'schemas.py': 'from __future__ import annotations\n\nfrom dataclasses import dataclass, field\nfrom typing import Any, Dict, List, Optional\n\n\n@dataclass\nclass QualitySummary:\n    foreground_ratio: float\n    n_components: int\n    present_classes: List[int] = field(default_factory=list)\n    flags: List[str] = field(default_factory=list)\n    mean_confidence: Optional[float] = None\n    mean_fg_confidence: Optional[float] = None\n\n\n@dataclass\nclass InferenceItem:\n    agent_item_id: str\n    plane: str\n    model_key: str\n    case_ref: str\n    figure_path: Optional[str] = None\n    quality: Optional[QualitySummary] = None\n    metadata: Dict[str, Any] = field(default_factory=dict)\n\n\n@dataclass\nclass AgentDecision:\n    agent_item_id: str\n    agent_status: str\n    review_priority: str\n    agent_reasons: List[str]\n    recommended_action: str\n    human_review_required: bool = True\n', 'quality.py': 'from __future__ import annotations\n\nfrom typing import Any, Dict, List, Optional\nimport ast\nimport math\nimport numpy as np\n\n\ndef parse_list_like(value: Any) -> List[Any]:\n    """Parsea valores tipo list guardados en CSV: [], [\'a\'], [1, 2, 3]."""\n    if value is None:\n        return []\n    if isinstance(value, list):\n        return value\n    if isinstance(value, tuple):\n        return list(value)\n    try:\n        if isinstance(value, float) and math.isnan(value):\n            return []\n    except Exception:\n        pass\n\n    text = str(value).strip()\n    if text in ("", "nan", "None", "[]"):\n        return []\n\n    try:\n        parsed = ast.literal_eval(text)\n        if isinstance(parsed, list):\n            return parsed\n        if isinstance(parsed, tuple):\n            return list(parsed)\n        return [parsed]\n    except Exception:\n        if "," in text:\n            return [x.strip() for x in text.split(",") if x.strip()]\n        return [text]\n\n\ndef to_float(value: Any, default: Optional[float] = None) -> Optional[float]:\n    try:\n        if value is None:\n            return default\n        x = float(value)\n        if math.isnan(x):\n            return default\n        return x\n    except Exception:\n        return default\n\n\ndef compute_quality_flags_from_values(\n    *,\n    foreground_ratio: Optional[float],\n    n_components: Optional[int],\n    present_classes: Optional[List[int]],\n    mean_confidence: Optional[float],\n    mean_fg_confidence: Optional[float],\n    plane: str,\n) -> List[str]:\n    flags: List[str] = []\n    fg = foreground_ratio if foreground_ratio is not None else 0.0\n    comps = int(n_components or 0)\n    classes = present_classes or []\n\n    if fg < 0.002:\n        flags.append("foreground_muy_bajo")\n    if plane == "axial" and fg > 0.25:\n        flags.append("foreground_muy_alto")\n    if plane == "sagittal" and fg > 0.45:\n        flags.append("foreground_muy_alto")\n    if not classes:\n        flags.append("sin_clases_no_fondo")\n\n    if plane == "axial" and comps > 10:\n        flags.append("muchos_componentes")\n    if plane == "sagittal" and comps > 10:\n        flags.append("muchos_componentes")\n\n    if mean_confidence is not None and mean_confidence < 0.70:\n        flags.append("baja_confianza_media")\n    if mean_fg_confidence is not None and mean_fg_confidence < 0.75:\n        flags.append("baja_confianza_foreground")\n\n    return flags\n\n\ndef useful_classes_for_plane(plane: str) -> List[int]:\n    """Clases útiles para métricas del MVP.\n\n    Axial excluye raw_0 porque E11 lo documentó como minoritario/problemático.\n    Sagital usa clases 1,2,3.\n    """\n    if plane == "axial":\n        return [2, 3, 4, 5]\n    if plane == "sagittal":\n        return [1, 2, 3]\n    return []\n', 'agent.py': 'from __future__ import annotations\n\nfrom typing import Any, Dict, List\nimport pandas as pd\n\nfrom .quality import parse_list_like, to_float\n\n\nEXPECTED_MODEL_BY_PLANE = {\n    "axial": "axial_t2_alkafri",\n    "sagittal": "sagittal_spider",\n}\n\n\ndef expected_model_for_plane(plane: str) -> str:\n    return EXPECTED_MODEL_BY_PLANE.get(str(plane).strip().lower(), "unknown")\n\n\ndef evaluate_agent_item(row: Dict[str, Any]) -> Dict[str, Any]:\n    """Reglas de orquestación inspiradas en E14.\n\n    El agente no toma decisiones clínicas. Solo asigna prioridad de revisión\n    y deja razones trazables.\n    """\n    plane = str(row.get("plane", "")).strip().lower()\n    model_key = str(row.get("model_key", "")).strip()\n    expected_model = expected_model_for_plane(plane)\n\n    flags = parse_list_like(row.get("flags", []))\n    n_components = int(to_float(row.get("n_components"), 0) or 0)\n    mean_fg_conf = to_float(row.get("mean_fg_confidence"), None)\n    fg_ratio = to_float(row.get("foreground_ratio"), None)\n\n    reasons: List[str] = []\n\n    if expected_model != "unknown" and model_key != expected_model:\n        reasons.append(f"modelo inesperado: esperado={expected_model}, recibido={model_key}")\n\n    for flag in flags:\n        reasons.append(f"flag E13: {flag}")\n\n    if plane == "axial" and n_components > 10:\n        reasons.append("muchos componentes en máscara axial")\n\n    if plane == "sagittal" and n_components > 10:\n        reasons.append("componentes múltiples esperables; revisar continuidad anatómica")\n\n    if mean_fg_conf is not None and mean_fg_conf < 0.75:\n        reasons.append("confianza baja en foreground")\n\n    if fg_ratio is not None and fg_ratio < 0.002:\n        reasons.append("foreground muy bajo")\n    if fg_ratio is not None and plane == "axial" and fg_ratio > 0.25:\n        reasons.append("foreground axial alto")\n    if fg_ratio is not None and plane == "sagittal" and fg_ratio > 0.45:\n        reasons.append("foreground sagital alto")\n\n    unique_reasons = []\n    for r in reasons:\n        if r not in unique_reasons:\n            unique_reasons.append(r)\n\n    high_priority = (\n        any("modelo inesperado" in r for r in unique_reasons)\n        or any("confianza baja en foreground" in r for r in unique_reasons)\n        or any("foreground muy bajo" in r for r in unique_reasons)\n    )\n\n    medium_priority = bool(unique_reasons)\n\n    if high_priority:\n        status = "requiere_revision_prioritaria"\n        priority = "alta"\n        action = "Revisar overlay y considerar repetir preprocesamiento/inferencia antes de usar el resultado."\n    elif medium_priority:\n        status = "requiere_revision_con_atencion"\n        priority = "media"\n        action = "Revisar visualmente el overlay; si la anatomía es coherente, puede pasar a validación profesional."\n    else:\n        status = "listo_para_revision_estandar"\n        priority = "baja"\n        action = "Enviar a revisión profesional estándar como resultado asistido por IA."\n\n    return {\n        "expected_model": expected_model,\n        "agent_status": status,\n        "review_priority": priority,\n        "agent_reasons": unique_reasons,\n        "recommended_action": action,\n        "human_review_required": True,\n    }\n\n\ndef build_agent_decisions(worklist_df: pd.DataFrame) -> pd.DataFrame:\n    rows = []\n    for _, row in worklist_df.iterrows():\n        base = row.to_dict()\n        decision = evaluate_agent_item(base)\n        rows.append({**base, **decision})\n    return pd.DataFrame(rows)\n\n\ndef summarize_agent_decisions(decisions_df: pd.DataFrame) -> Dict[str, Any]:\n    summary: Dict[str, Any] = {\n        "total_items": int(len(decisions_df)),\n        "plane_distribution": decisions_df["plane"].value_counts().to_dict() if "plane" in decisions_df else {},\n        "priority_distribution": decisions_df["review_priority"].value_counts().to_dict() if "review_priority" in decisions_df else {},\n        "status_distribution": decisions_df["agent_status"].value_counts().to_dict() if "agent_status" in decisions_df else {},\n    }\n\n    if "mean_fg_confidence" in decisions_df:\n        summary["mean_fg_confidence"] = float(pd.to_numeric(decisions_df["mean_fg_confidence"], errors="coerce").mean())\n\n    if "dice_macro_useful_classes" in decisions_df:\n        summary["mean_dice_macro_useful_classes"] = float(\n            pd.to_numeric(decisions_df["dice_macro_useful_classes"], errors="coerce").mean()\n        )\n\n    return summary\n', 'reporting.py': 'from __future__ import annotations\n\nfrom pathlib import Path\nfrom typing import Any, Dict\nimport json\nimport pandas as pd\n\n\ndef write_json(path: Path, data: Dict[str, Any]) -> Path:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")\n    return path\n\n\ndef write_case_report(path: Path, row: Dict[str, Any]) -> Path:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    md = f"""# Reporte de caso IA - {row.get(\'agent_item_id\')}\n\n## Identificación\n\n- Plano: {row.get(\'plane\')}\n- Caso: {row.get(\'case_ref\')}\n- Modelo: {row.get(\'model_key\')}\n\n## Calidad\n\n- Foreground ratio: {row.get(\'foreground_ratio\')}\n- Componentes: {row.get(\'n_components\')}\n- Confianza media: {row.get(\'mean_confidence\')}\n- Confianza foreground: {row.get(\'mean_fg_confidence\')}\n- Flags: {row.get(\'flags\')}\n\n## Decisión del agente\n\n- Estado: {row.get(\'agent_status\')}\n- Prioridad: {row.get(\'review_priority\')}\n- Razones: {row.get(\'agent_reasons\')}\n- Acción recomendada: {row.get(\'recommended_action\')}\n\n## Política\n\nEste resultado es asistido por IA y requiere revisión profesional.\n"""\n    path.write_text(md, encoding="utf-8")\n    return path\n\n\ndef build_markdown_summary(summary: Dict[str, Any]) -> str:\n    return f"""# Resumen del agente IA\n\nTotal de ítems: {summary.get(\'total_items\')}\n\nDistribución por plano: {summary.get(\'plane_distribution\')}\n\nDistribución por prioridad: {summary.get(\'priority_distribution\')}\n\nDistribución por estado: {summary.get(\'status_distribution\')}\n\nConfianza foreground media: {summary.get(\'mean_fg_confidence\')}\n\nDice útil medio: {summary.get(\'mean_dice_macro_useful_classes\')}\n\nEl agente funciona como apoyo y no reemplaza la revisión profesional.\n"""\n', 'api.py': 'from __future__ import annotations\n\nfrom pathlib import Path\nimport pandas as pd\n\nfrom fastapi import FastAPI, HTTPException\n\nfrom .settings import get_settings, MODEL_REGISTRY\nfrom .agent import build_agent_decisions, summarize_agent_decisions\nfrom .reporting import build_markdown_summary\n\napp = FastAPI(\n    title="PFI Lumbar MRI AI Service",\n    description="Servicio IA para segmentación asistida y orquestación de revisión profesional.",\n    version="0.1.0",\n)\n\n\n@app.get("/health")\ndef health():\n    settings = get_settings()\n    return {\n        "status": "ok",\n        "pfi_root": str(settings.pfi_root),\n        "human_review_required": True,\n    }\n\n\n@app.get("/models")\ndef models():\n    settings = get_settings()\n    return {\n        "models": MODEL_REGISTRY,\n        "paths": {\n            "sagittal_model_path": str(settings.sagittal_model_path),\n            "axial_model_path": str(settings.axial_model_path),\n        },\n    }\n\n\n@app.get("/agent/worklist")\ndef agent_worklist():\n    settings = get_settings()\n    worklist_path = settings.e14_results_root / "E14_agent_worklist.csv"\n    if not worklist_path.exists():\n        raise HTTPException(status_code=404, detail=f"No existe {worklist_path}")\n\n    df = pd.read_csv(worklist_path)\n    return {\n        "rows": int(len(df)),\n        "items": df.to_dict(orient="records"),\n    }\n\n\n@app.get("/agent/report")\ndef agent_report():\n    settings = get_settings()\n    worklist_path = settings.e14_results_root / "E14_agent_worklist.csv"\n    metrics_path = settings.e14_results_root / "E14_agent_metrics_summary.csv"\n\n    if not worklist_path.exists():\n        raise HTTPException(status_code=404, detail=f"No existe {worklist_path}")\n\n    worklist = pd.read_csv(worklist_path)\n    decisions = build_agent_decisions(worklist)\n\n    if metrics_path.exists():\n        metrics = pd.read_csv(metrics_path)\n        if "agent_item_id" in metrics.columns:\n            decisions = decisions.merge(metrics, on=["agent_item_id", "plane", "case_ref"], how="left")\n\n    summary = summarize_agent_decisions(decisions)\n\n    return {\n        "summary": summary,\n        "markdown": build_markdown_summary(summary),\n        "items": decisions.to_dict(orient="records"),\n    }\n', 'README.md': '# PFI AI Service\n\nServicio Python inicial para el MVP de análisis asistido de RM lumbar.\n\n## Objetivo\n\nEste paquete extrae la lógica consolidada de los notebooks E13/E14 hacia módulos reutilizables:\n\n- configuración centralizada,\n- contratos simples de datos,\n- quality flags,\n- reglas del agente/orquestador,\n- generación de reportes,\n- API FastAPI inicial.\n\n## Alcance\n\nEste servicio no reemplaza la revisión profesional. Su rol es:\n\n1. organizar resultados de inferencia,\n2. calcular flags de calidad,\n3. priorizar revisión,\n4. generar reportes trazables para validación humana.\n\n## Estructura\n\n```text\nai_service/\n  pfi_ai_service/\n    settings.py\n    schemas.py\n    quality.py\n    agent.py\n    reporting.py\n    api.py\n  requirements-ai-service.txt\n  README.md\n```\n\n## Endpoints propuestos\n\n```text\nGET /health\nGET /models\nGET /agent/worklist\nGET /agent/report\n```\n\n## Ejecución local futura\n\n```bash\ncd ai_service\npip install -r requirements-ai-service.txt\nuvicorn pfi_ai_service.api:app --reload --host 0.0.0.0 --port 8000\n```\n\n## Integración con backend Java/Spring Boot\n\nEl backend puede invocar este servicio Python por HTTP o mediante un job interno, manteniendo la lógica pesada de IA separada del backend transaccional.\n', 'requirements-ai-service.txt': 'fastapi>=0.110\nuvicorn[standard]>=0.29\npandas>=2.0\nnumpy>=1.24\npydantic>=2.0\n'}

# Escribir archivos
for rel_path, content in SERVICE_FILES.items():
    if rel_path in ["README.md", "requirements-ai-service.txt"]:
        out_path = SERVICE_ROOT / rel_path
    else:
        out_path = PKG_ROOT / rel_path

    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(content, encoding="utf-8")
    print("Wrote:", out_path)

# Inventario
inventory_rows = []
for p in sorted(SERVICE_ROOT.rglob("*")):
    if p.is_file():
        inventory_rows.append({
            "relative_path": str(p.relative_to(REPO_ROOT)),
            "size_bytes": p.stat().st_size,
        })

inventory_df = pd.DataFrame(inventory_rows)
inventory_csv = E15_ROOT / "E15_service_file_inventory.csv"
inventory_df.to_csv(inventory_csv, index=False)

display(inventory_df)
print("Inventory:", inventory_csv)


## 2. Smoke test del paquete generado

Este test no levanta el servidor. Verifica que el paquete importe correctamente y que las reglas del agente puedan operar sobre los outputs reales de E14.


In [ ]:
# Smoke test de imports y reglas del agente

if str(SERVICE_ROOT) not in sys.path:
    sys.path.insert(0, str(SERVICE_ROOT))

import pfi_ai_service
from pfi_ai_service.settings import get_settings, MODEL_REGISTRY
from pfi_ai_service.agent import build_agent_decisions, summarize_agent_decisions
from pfi_ai_service.reporting import build_markdown_summary, write_json

settings = get_settings()

print("Package:", pfi_ai_service.__version__)
print("Settings pfi_root:", settings.pfi_root)
print("Models:", MODEL_REGISTRY.keys())

WORKLIST_PATH = E14_ROOT / "E14_agent_worklist.csv"
METRICS_PATH = E14_ROOT / "E14_agent_metrics_summary.csv"
FULL_E14_PATH = E14_ROOT / "E14_agent_decisions_with_metrics.csv"

for p in [WORKLIST_PATH, METRICS_PATH]:
    print(p, "->", p.exists())
    assert p.exists(), f"Falta output E14 requerido: {p}"

worklist_df = pd.read_csv(WORKLIST_PATH)
metrics_df = pd.read_csv(METRICS_PATH)

decisions_df = build_agent_decisions(worklist_df)

# Merge con métricas si están disponibles
merge_keys = [c for c in ["agent_item_id", "plane", "case_ref"] if c in decisions_df.columns and c in metrics_df.columns]
if merge_keys:
    decisions_with_metrics_df = decisions_df.merge(metrics_df, on=merge_keys, how="left")
else:
    decisions_with_metrics_df = decisions_df.copy()

summary = summarize_agent_decisions(decisions_with_metrics_df)

smoke_decisions_csv = E15_ROOT / "E15_smoke_agent_decisions.csv"
decisions_with_metrics_df.to_csv(smoke_decisions_csv, index=False)

smoke_report = {
    "notebook": "23_E15_backend_mvp_translation",
    "goal": "smoke test generated ai_service package using E14 outputs",
    "package_version": pfi_ai_service.__version__,
    "service_root": str(SERVICE_ROOT),
    "worklist_path": str(WORKLIST_PATH),
    "metrics_path": str(METRICS_PATH),
    "smoke_decisions_csv": str(smoke_decisions_csv),
    "summary": summary,
    "checks": {
        "package_import_ok": True,
        "worklist_rows": int(len(worklist_df)),
        "decisions_rows": int(len(decisions_df)),
        "metrics_rows": int(len(metrics_df)),
        "has_priority_distribution": bool(summary.get("priority_distribution")),
        "human_review_required": True,
    },
    "decision": "ai_service_package_smoke_test_passed",
}

smoke_report_path = E15_ROOT / "E15_smoke_test_report.json"
smoke_report_path.write_text(json.dumps(smoke_report, indent=2, ensure_ascii=False), encoding="utf-8")

print(json.dumps(smoke_report, indent=2, ensure_ascii=False))
display(decisions_with_metrics_df.head(12))
print("Smoke report:", smoke_report_path)


## 3. Generar documentación E15 y prompt para Codex

Esta celda deja documentación para tesis/backlog y un prompt listo para pedirle a Codex que integre el servicio al backend/MVP.


In [ ]:
summary_md = build_markdown_summary(summary)

e15_md = f"""# E15 - Traducción a backend/MVP

## Objetivo

E15 traduce la cadena funcional de notebooks hacia una estructura inicial de servicio Python para MVP.

No se entrenaron modelos nuevos. El foco fue convertir la lógica de E13/E14 en módulos reutilizables para una futura integración con backend.

## Archivos generados

Carpeta principal:

```text
{SERVICE_ROOT}
```

Archivos:

```text
{chr(10).join("- " + r["relative_path"] for _, r in inventory_df.iterrows())}
```

## Smoke test

Se cargaron las salidas de E14:

- `{WORKLIST_PATH}`
- `{METRICS_PATH}`

Resumen del smoke test:

```json
{json.dumps(summary, indent=2, ensure_ascii=False)}
```

## Decisión metodológica

El servicio generado representa el paso desde experimentación en Colab hacia arquitectura de producto.

La política se mantiene:

- el sistema es de apoyo a la decisión;
- requiere revisión profesional;
- no reemplaza diagnóstico clínico;
- conserva trazabilidad de flags, prioridades y razones del agente.

## Próximo paso

Integrar este servicio con backend Spring Boot o generar un prototipo FastAPI ejecutable.
"""

e15_md_path = DOCS_ROOT / "E15_backend_mvp_translation_conclusion.md"
e15_md_path.write_text(e15_md, encoding="utf-8")

codex_prompt = f"""# Prompt para Codex - Integración backend/MVP del servicio IA

Contexto:
Estoy trabajando en un PFI de Ingeniería Informática sobre una plataforma de apoyo al análisis estructural de RM lumbar mediante segmentación asistida por IA. Ya tengo una cadena funcional en Colab:

- E10: modelo axial T2 final.
- E11: mapeo y documentación de clases axiales.
- E12: modelo sagital final consolidado.
- E13: pipeline común de inferencia multiplanar.
- E14: agente/orquestador IA con quality flags, prioridades y reportes.

Ahora generé en el repo la carpeta:

```text
ai_service/
```

Objetivo:
Convertir `ai_service` en un microservicio Python/FastAPI integrable con el backend principal.

Tareas:
1. Revisar la estructura `ai_service/pfi_ai_service`.
2. Asegurar que `settings.py` permita configurar `PFI_ROOT` por variable de entorno.
3. Probar endpoints:
   - `GET /health`
   - `GET /models`
   - `GET /agent/worklist`
   - `GET /agent/report`
4. Agregar instrucciones de ejecución local en README.
5. Preparar un Dockerfile si tiene sentido.
6. Proponer cómo Spring Boot debería invocar el servicio Python.
7. Mantener explícita la política human-in-the-loop: el servicio no reemplaza la revisión profesional.

Criterios de aceptación:
- El servicio levanta con `uvicorn pfi_ai_service.api:app`.
- `/health` responde OK.
- `/agent/report` devuelve resumen de prioridades usando outputs E14.
- El código queda modular y preparado para agregar endpoints reales de inferencia después.
"""

codex_prompt_path = DOCS_ROOT / "E15_backend_mvp_codex_prompt.md"
codex_prompt_path.write_text(codex_prompt, encoding="utf-8")

final_report = {
    "notebook": "23_E15_backend_mvp_translation",
    "goal": "translate E13/E14 notebook logic into backend/MVP Python service structure",
    "service_root": str(SERVICE_ROOT),
    "inventory_csv": str(inventory_csv),
    "smoke_report_json": str(smoke_report_path),
    "docs": {
        "conclusion_md": str(e15_md_path),
        "codex_prompt_md": str(codex_prompt_path),
    },
    "summary": summary,
    "decision": "backend_mvp_translation_ready_for_codex_or_backend_integration",
}

final_report_path = E15_ROOT / "E15_backend_mvp_translation_report.json"
final_report_path.write_text(json.dumps(final_report, indent=2, ensure_ascii=False), encoding="utf-8")

print("E15 conclusion:", e15_md_path)
print("Codex prompt:", codex_prompt_path)
print("Final report:", final_report_path)
print(json.dumps(final_report, indent=2, ensure_ascii=False))


## 4. Comandos sugeridos para commitear desde Colab

Después de revisar que E15 corrió bien, commitear los archivos generados:

```bash
%cd /content/drive/MyDrive/PFI_MVP/repo
!git status
!git add ai_service/ notebooks/23_E15_backend_mvp_translation.ipynb
!git commit -m "Add E15 backend MVP AI service scaffold"
!git push
```

También guardar el notebook desde Colab a GitHub si estás trabajando con la UI.
